In [1]:
import scanpy as sc
import squidpy as sq
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import anndata as ad
from scipy import sparse
import anndata as ad
import json

/home/lucia/miniforge3/envs/masters_env_minimal/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/lucia/miniforge3/envs/masters_env_minimal/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/lucia/miniforge3/envs/masters_env_minimal/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead

In [2]:
adata = ad.read_h5ad("BRCA_preprocessed.h5ad", backed=None)  # load the preprocessed data
library_id='Visium_Human_Breast_Cancer'
adata

AnnData object with n_obs × n_vars = 4809 × 19808
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts_umi', 'log1p_total_counts_umi', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'tissue_status', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts_genes', 'log1p_total_counts_genes', 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'moranI', 'pca', 'spatial', 'spatial_neighbors'
    obsm: 'X_pca', 'spatial'
    varm: 'PCs'
    layers: 'log1p', 'log1p_norm'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [3]:
%cd './src'

import importlib
import ssgsea
import ssgsea_plots
import multimodal

importlib.reload(ssgsea)
importlib.reload(ssgsea_plots)
importlib.reload(multimodal)

%cd ..

/home/lucia/repos/Thesis_final/src
/home/lucia/repos/Thesis_final


In [4]:
# image segmentation pipeline, already ran
# adata_img = multimodal.run_img_segmentation(
#                 adata = adata,
#                 output_dir = "segmentation_results",
#                 sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
#                 superpixel = "slic",
#                 res = "hires"
#                 )

In [5]:
adata_img = ad.read_h5ad("segmentation_results/adata_img_segmentation_slic.h5ad")

In [6]:
adata_img

AnnData object with n_obs × n_vars = 4847 × 24387
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts_umi', 'log1p_total_counts_umi', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'tissue_status', 'n_counts', 'segmentation_slic_sigma0', 'segmentation_slic_sigma0.5', 'segmentation_slic_sigma1', 'segmentation_slic_sigma2', 'segmentation_slic_sigma4', 'segmentation_slic_sigma6', 'segmentation_slic_sigma8', 'segmentation_slic_sigma10'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts_genes', 'log1p_total_counts_genes', 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'pca', 'spatial'
    obsm: 'X_pca', 'spatial'
    varm: 'PCs'
    layers: 'log1p', 'log1p_norm'
    obsp: 'segmentation_slic_sigma0.5_dist', 'segmentation_slic_sigma0.5_dist_scaled', 'segmentation_slic_sigma

In [7]:
# # run ssgsea, get df
# ssgsea = ssgsea.ssGSEA()
# results = ssgsea.compute_ssgsea(adata, 'h.all.v2026.1.Hs.json', layer=None)

In [8]:
ssgsea_df = pd.read_csv('NES.csv', index_col=0)

In [9]:
multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    distance_gexpr='correlation',
    use_pca=True,
    modality_combinations = [True, False], # how I want to combine gexpr and gsets
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1), (0.25, 0.75), (0.75, 0.25)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_gexpr_correlation"
    )

: 

In [ ]:
%cd './src'

import importlib
import ssgsea
import ssgsea_plots
import multimodal

importlib.reload(ssgsea)
importlib.reload(ssgsea_plots)
importlib.reload(multimodal)

%cd ..

In [ ]:
# plots for EMT

results_adata_path = Path('multimodal_results_EMT/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    ssgsea_plots.show_results(ssgsea_df[['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']], 
                              gene_set='HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 
                              results_adata_dir=inp, output_dir=out)